In [96]:
import chromadb
from chromadb.utils import embedding_functions 
import json

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
 
collection = client.get_or_create_collection(
    name="my_docs",
    embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
)

In [ ]:

def answer_question(user_query, collection, top_k=3):
    """
    Повний RAG пайплайн: пошук → контекст → відповідь
    """
    
    results = collection.query(
        query_texts=[user_query],
        n_results=top_k
    )
    
    chunks = results['documents'][0]
    sources = results['metadatas'][0]
    distances = results['distances'][0]
    
    context_parts = []
    for chunk, meta, dist in zip(chunks, sources, distances):
      
        context_parts.append(
            f"[ДЖЕРЕЛО: {meta['source']}, СТОРІНКА: {meta['page']}, РЕЛЕВАНТНІСТЬ: {1-dist:.2f}]\n{chunk}"
        )
    
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f""" 
            You are an assistant who responds ONLY based on the context provided.
        RULES:
        1. Answer ONLY based on the context below.
        2. If the information is NOT in the context, honestly say “I don't know” or “This information is not included in the provided documents.”
        3. DO NOT make up information that is not in the context.
        4. If you answer, cite your source.

        CONTEXT (taken from documents):
        {context}

        USER QUESTION:
        {user_query}

        ANSWER (based on the context):
    """
         
    # query for llama3.2 model
    try:
        import ollama 
        response = ollama.chat(
            model = 'llama3.2',
            
            messages=[
                {"role": "system", "content": "You are an assistant. Respond only based on the context provided. If you don't know, say 'I don't know.'"},
                {"role": "user", "content": prompt}
            ],
            options = {
                'temperature': 0.3, # To more accuracy answer
                'num_predict': 500  # To more better think, but longer
            }   
        )
        answer = response['message']['content']
    except:
        print(f"Ollama недоступний")
        answer = "[ПОТРІБНА LLM МОДЕЛЬ]"
     
    return {
        'answer': answer,
        'sources': sources,
        'chunks': chunks,
        'context': context,
        'prompt': prompt
    }

In [99]:
if __name__ == "__main__":
    user_query = "NLP what is that!"
    
    result = answer_question(user_query, collection, top_k=3)
     
    print(f"Questions: {user_query}")
 
    print("\nAnswer: ")
    print(result['answer'])
    
    print("\n" + "="*70)
    print("Source: ")
    for i, source in enumerate(result['sources'], 1):
        print(f"{i}. {source['source']} (стор. {source['page']})")

Questions: NLP what is that!

Answer: 
NLP stands for Natural Language Processing. According to the provided documents, NLP has long helped extract signals from language and has evolved over time, including the development of embeddings. The document also mentions that NLP in finance has become increasingly important due to the explosion of text data and the need for firms to integrate both classical NLP and Large Language Models (LLMs) to stay competitive.

Source: rf_aiinassetmanagement_practitioner-briefs_07_naturallanguageprocessing_online.pdf, "Natural Language Processing" section.

Source: 
1. rf_aiinassetmanagement_practitioner-briefs_07_naturallanguageprocessing_online.pdf (стор. 1)
2. 2025.unlp-1.18.pdf (стор. 10)
3. rf_aiinassetmanagement_practitioner-briefs_07_naturallanguageprocessing_online.pdf (стор. 1)
